# Problem wyboru punktów do pomiaru warunków pogodowych
### Wstęp i analiza problemu
W analizie ruchu komunikacji miejskiej bardzo ważnym elementem wpływającym na jej opóźnienia są warunki pogodowe. Wykorzystując [API pogodowe](https://open-meteo.com/) jestem w stanie otrzymać informacje o warunkach atmosferycznych w miejscu o wybranych współrzędnych. Mimo że API pozwala w jednym zapytaniu na otrzymanie danych o wielu wybranych punktach, muszą one być znane w momencie wysyłania zapytania. Niepraktycznym byłoby za każdym odpytaniem o położenie autobusów, odpytywać również o pogodę dla każdego z położeń. Dokładność pomiarów pogody wynosi w najlepszym przypadku ok. 1km. Pytając o pogodę równie często, co API urzędu miasta mające aktualizacje co ok. 15 sekund, ryzykuję bycie zablokowanym przez domenę open-meteo z powodu zalania jej wielką falą zapytań o praktycznie to samo.

Rozwiązanie problemu czasowego jest dość proste - odpytywać co np. pół godziny. Pogoda nie jest zjawiskiem tak gwałtownym, żeby zmienić się diametralnie w przeciągu minut (przynajmniej nie w Warszawie). Kwestia rozmieszczenia punktów pomiarowych natomiast jest już nieco bardziej złożona. W zależności od wybranych do pomiarów linii, autobusy będą poruszać się w innych miejscach. Ponadto, w dzielnicach centralnych (np. Śródmieście, Ochota) będzie jeździć ich rzecz jasna znacznie więcej niż na obrzerzach (np. Ursus, Bielany). Zastosowanie równomiernie rozłożonej siatki na całym mieście mogłoby być z tego powodu nieoptymalne. Powstałyby punkty praktycznie nieużywane przy granicach miasta, a w centrum siatka byłaby zbyt rzadka. Potrzebny zatem jest sposób na wyznacznie odpowiedniej liczby punktów pomiarowych w odpowiednich miejscach.

W tym momencie na ratunek przychodzi uczenie maszynowe. Do rozwiązania tego problemu wykorzystam algorytm grupowania K-średnich. Jako grupę punktów określających trasy pojazdów wykorzystam współrzędne wszystkich przystanków na wszystkich trasach wszystkich analizowanych linii. Wynikiem grupowania będzie optymalna liczba centroidów, które będą wyznaczały punkty pomiarowe pogody. Algorytm K-średnich zapewni, że obszary o większym zagęszczeniu przystanków (a co za tym idzie o większej liczbie przejeżdżających tam pojazdów) zostaną opisane większą liczbą punktów pomiarowych, a obszary rzadziej odwiedzane będą reprezentowane w rozsądnej (minimalizującej dystanse do punktów wyznaczających centroid) odległości.

W tym notatniku przedstawiam proces obserwacji i badania doboru liczby i położeń punktów pomiarowych. Docelowo połączony proces wyboru został zaimplementowany w funkcji `wyznacz_punkty_pomiarowe_pogody()` w pliku `utils.py`.

### Implementacja
Zacznijmy od importu bibliotek i przygotowania informacji o przystankach:

In [ ]:
import json
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import find_dotenv, load_dotenv
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import src.kolektor_danych as kolektor_danych
from src.utils import DATA_DIR

linie = ['114', '116', '135', '138', '148', '157', '158', '185', '189', '500', '504', '509', '517', '523']

Poniższy blok należy wywołać tylko, gdy w `DATA_DIR` projektu nie znajdują się pliki z trasami wszystkich wyżej wymienionych linii lub brakuje pliku opisującego przstanki. Warto zakomentować sobie niepotrzebne w momencie pobierania fragmenty.

In [ ]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)
API_KEY = os.getenv('API_KEY')
if API_KEY is None:
    print("Nie znaleziono klucza api w pliku .env")
    exit(1)
if API_KEY == "":
    print("Klucz api w pliku .env jets pusty")
    exit(1)

kolektor_danych.stworz_baze_polozen_przystankow(API_KEY)
for linia in linie:
    kolektor_danych.stworz_trase_linii(API_KEY, linia)
    kolektor_danych.stworz_rozklad_linii(API_KEY, linia)

Kolejno wczytuję znajdujące się już na dysku dane o przystankach:

In [ ]:
wszystkie_przystanki = set()
pattern_str = r"trasa_(" + "|".join(linie) + r")\.json"
pattern = re.compile(pattern_str)
for filename in os.listdir(DATA_DIR):
    if pattern.match(filename):
        with open(DATA_DIR / filename) as f:
            trasa = json.load(f)
            for wariant in trasa['warianty_tras']:
                if len(wariant) <= 2:
                    continue
                wszystkie_przystanki.update(trasa['warianty_tras'][wariant].keys())
                
wspolrzedne = []
with open(DATA_DIR / 'przystanki.json') as f:
    przystanki = json.load(f)
    for przystanek_id in wszystkie_przystanki:
        lat = przystanki[przystanek_id]['lat']
        lon = przystanki[przystanek_id]['lon']
        wspolrzedne.append([lat, lon])

Teraz **najważniejsza część**, czyli proces wyboru położenia i liczby punktów. Na początku wybieram sobie jaki przedział liczby centroidów będę analizować oraz liczbę iteracji.

Jako metrykę porównującą wykorzystam metrykę sylwetki klastra.
Dla punktu $i \in C_i$ obliczam $a(i)$, średnią odległość punktu w klastrze od pozostałych punktów należących do tego klastra:
$$a(i) = \frac{1}{|C_i|-1} \sum_{j \in C_i, i \neq j} d(i,j)$$
gdzie $|C_i|$ to liczba punktów należących do klastra $C_i$, a $d(i,j)$ to odległość między punktami $i$ i $j$.

Kolejno, dla każdego punktu z klastra $C_i$ określam $b(i)$, czyli średnią odległość między punktem $i$ a wszystkimi punktami innego klastra $C_j$ (miara niepodobieństwa):
$$b(i) = \min_{j \neq i} \frac{1}{|C_j|} \sum_{l \in C_j} d(i,l)$$

Klaster z najmnijeszą miarą niepodobieństwa jest najbliższym klastrem punktowi $i$.

Mając te dwie miary można obliczyć wartość sylwetki dla punktu $i$:
$$s(i) = \begin{cases}
1 - \frac{a(i)}{b(i)} & \text{jeśli} & a(i) \lt b(i) \\
0 & \text{jeśli} & a(i) = b(i) \\
\frac{b(i)}{a(i)} - 1 & \text{jeśli} & a(i) \gt b(i) \\
\end{cases}
$$

Wartość sylwetki zawiera się w przedziale $[-1, 1]$. Znajdzie się bliżej $1$, gdy $a(i) \ll b(i)$, czyli średnia odległość od elementów tego klastra jest znacznie mniejsza niż średnia odległość od elementów najbliższego innego klastra. Bliżej wartości $-1$ natomiast, gdy $a(i) \gg b(i)$, czyli średnio punkt jest bliżej punktów innego klastra niż swojego.

Ze względu na losowość stanu początkowego algorytmu K-średnich (początkowych położeń centroidów) wyznaczać będę średnią wartość sylwetki z wielu prób. Zrobię to zarówno za pomocą wbudowanego w scikit-learn mechanizmu jak i dodatkowej własnej pętli.
1. Wbudowany w model grupowania parametr `n_init` określa ile razy algorytm zostanie uruchomiony z różnymi ziarnami losowania, a jako wynik grupowania zostanie podana próba o najmniejszej sumie kwadratów odległości wszystkich punktów od przypisanych im klastrów.
2. Grupowanie w ten sposób zostanie powtórzone wielokrotnie, a jako wynik do porównywania skuteczności wyboru danej liczby klastrów obliczana jest średnia arytmetyczna.

In [70]:
zakres_k_centroidow = range(5, 25)
wyniki_sylwetek = {k: [] for k in zakres_k_centroidow}

iteracji_usredniajacych = 30
n_init = 20

X = np.array(wspolrzedne)
for j in range(iteracji_usredniajacych):
    print(f"Próba {j+1}/{iteracji_usredniajacych}...", end='\r', flush=True)
    for k in zakres_k_centroidow:
        kmeans = KMeans(n_clusters=k, n_init=n_init, random_state=67).fit(X)
        wynik_sylwetki = silhouette_score(X, kmeans.labels_)
        wyniki_sylwetek[k].append(wynik_sylwetki)

srednie_sylwetki = {k: np.mean(sylwetki) for k, sylwetki in wyniki_sylwetek.items()}
najlepsze_k = max(srednie_sylwetki, key=lambda k: srednie_sylwetki[k])
print(f"Optymalna liczba klastrów: {najlepsze_k} (średnia wartość sylwetki: {srednie_sylwetki[najlepsze_k]})")

fig_sylwetka = px.line(
    x=list(zakres_k_centroidow), 
    y=srednie_sylwetki, 
    markers=True,
    title=f'Analiza sylwetkowa (średnia z {iteracji_usredniajacych} iteracji uśredniających wynik najlepszych {n_init} prób dla każdego k)',
    labels={'x': 'Liczba klastrów (k)', 'y': 'Średnia wartość sylwetki'}
)

fig_sylwetka.update_xaxes(dtick=1)
fig_sylwetka.show()

Optymalna liczba klastrów: 10 (średnia wartość sylwetki: 0.4417391904991008)


Na wykresie widać dla jakiej liczby klastrów średnia wartość sylwetki jest największa (najkorzystniejsza). Użycie wielokrotnego grupowania nie rozwiązuje w 100% problemu losowości tej metody, ale znacznie go ogranicza. Oczywiście im więcej prób zostanie wykonane tym większą mamy pewność, że wynik będzie optymalny, ale to za cenę czasu.

### Wizualizacja wyników
Poniżej na koniec można zobaczyć wizualizację wyniku grupowania. Na czarno zaznaczone są centroidy, natomiast różnymi kolorami przystanki należące do wspólnego klastra.

In [69]:
X = np.array(wspolrzedne)
kmeans = KMeans(n_clusters=najlepsze_k).fit(X)

df_punkty = pd.DataFrame(X, columns=['lat', 'lon'])
df_punkty['Typ'] = 'Punkt danych'
df_punkty['Klaster'] = kmeans.labels_.astype(str)
df_punkty['Rozmiar'] = 3

df_centroidy = pd.DataFrame(kmeans.cluster_centers_, columns=['lat', 'lon'])
df_centroidy['Typ'] = 'Centroid'
df_centroidy['Klaster'] = 'Centroid'
df_centroidy['Rozmiar'] = 9

df_mapa = pd.concat([df_punkty, df_centroidy], ignore_index=True)

fig = px.scatter_map(
    df_mapa, 
    lat="lat", 
    lon="lon", 
    color="Klaster",
    zoom=10, 
    size="Rozmiar",
    size_max=9,
    color_discrete_map={'Centroid': 'black'},
    width=900,
    height=900,
    color_discrete_sequence=px.colors.qualitative.Light24,
)

margines = 0.06

fig.update_layout(
    map=dict(
        bounds=dict(
            west=df_mapa['lon'].min() - margines,
            east=df_mapa['lon'].max() + margines,
            south=df_mapa['lat'].min() - margines,
            north=df_mapa['lat'].max() + margines
        )
    )
)

fig.show()